In [1]:
import sys, json
sys.path.append("..")
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import DataLoader
from src.tokenizer import CharTokenizer
from src.dataset import SQLDataset, collate
from src.model import GPTConfig, TinyGPT

def load(name):
    return [json.loads(line) for line in open(f"../data/raw/{name}.jsonl", encoding="utf-8")]

train = load("train")
tok = CharTokenizer.build([r["question"] for r in train] + [r["sql"] for r in train])
print("vocab size:", tok.vocab_size)

vocab size: 73


In [2]:
config = GPTConfig(vocab_size=tok.vocab_size)
model = TinyGPT(config)

loader = DataLoader(SQLDataset(train, tok), batch_size=4, shuffle=True,
                    collate_fn=lambda b: collate(b, tok.pad_id))
xb, yb = next(iter(loader))

out = model(xb)
print("input :", xb.shape)
print("output:", out.shape)

n_params = sum(p.numel() for p in model.parameters())
print("parameters:", n_params)

input : torch.Size([4, 264])
output: torch.Size([4, 264, 128])
parameters: 44160
